In [1]:
# Hücre 1: Drive bağlantısı ve proje klasör yapısı
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/SentinelAI"
folders = ["data/raw", "data/processed", "data/mitre", "models", "reports", "notebooks"]

for f in folders:
    os.makedirs(f"{BASE}/{f}", exist_ok=True)

os.chdir(BASE)
print(" Çalışma dizini:", os.getcwd())
print("\n Klasör yapısı:")
for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

Mounted at /content/drive
 Çalışma dizini: /content/drive/MyDrive/SentinelAI

 Klasör yapısı:
SentinelAI/
  data/
    raw/
    processed/
    mitre/
  models/
  reports/
  notebooks/
  __pycache__/


In [2]:
# Hücre 2: Sentetik log verisi üretimi
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

NORMAL_IPS = [f"192.168.1.{i}" for i in range(10, 40)]
USERS = ["admin", "alice", "bob", "carol", "dave", "eve"]
ATTACK_IP = "192.168.1.45"   # brute-force saldırganı

def generate_normal_logs(n=2000):
    """Normal kullanıcı davranışı: gündüz saatleri, az deneme, başarılı girişler"""
    rows = []
    for _ in range(n):
        ts = datetime(2026, 11, 17) + timedelta(seconds=np.random.randint(0, 86400))
        rows.append({
            "timestamp": ts,
            "ip": np.random.choice(NORMAL_IPS),
            "event": np.random.choice(
                ["Login Success", "File Access", "Logout", "Login Success"],
                p=[0.4, 0.3, 0.2, 0.1]
            ),
            "user": np.random.choice(USERS),
            "attempts": np.random.randint(1, 4),
            "true_label": 0,   # 0 = normal
        })
    return rows

def generate_bruteforce_logs(n=100):
    """Saldırı: gece 02:00 civarı, aynı IP, çok deneme, hep başarısız"""
    rows = []
    for _ in range(n):
        ts = datetime(2026, 11, 17, 2, 0) + timedelta(seconds=np.random.randint(0, 1800))
        rows.append({
            "timestamp": ts,
            "ip": ATTACK_IP,
            "event": "Failed Login",
            "user": "admin",
            "attempts": np.random.randint(20, 40),
            "true_label": 1,   # 1 = saldırı
        })
    return rows

df = pd.DataFrame(generate_normal_logs() + generate_bruteforce_logs())
df = df.sort_values("timestamp").reset_index(drop=True)

# Drive'a kaydet
out_path = f"{BASE}/data/raw/synthetic_logs.csv"
df.to_csv(out_path, index=False)

print(f" Toplam log: {len(df)}")
print(f" Normal: {(df['true_label']==0).sum()}  |  Saldırı: {(df['true_label']==1).sum()}")
print(f" Kaydedildi: {out_path}")
print("\n İlk 5 satır:")
df.head()

 Toplam log: 2100
 Normal: 2000  |  Saldırı: 100
 Kaydedildi: /content/drive/MyDrive/SentinelAI/data/raw/synthetic_logs.csv

 İlk 5 satır:


,timestamp,ip,event,user,attempts,true_label
0,2026-11-17 00:01:00,192.168.1.31,Login Success,carol,1,0
1,2026-11-17 00:01:06,192.168.1.25,Login Success,admin,2,0
2,2026-11-17 00:02:16,192.168.1.15,File Access,bob,3,0
3,2026-11-17 00:02:28,192.168.1.36,Login Success,alice,1,0
4,2026-11-17 00:03:27,192.168.1.19,Login Success,admin,3,0


In [3]:
# Hücre 3: Log parsing ve özellik mühendisliği
def parse_logs(path):
    df = pd.read_csv(path, parse_dates=["timestamp"])

    # Zaman özellikleri
    df["hour"] = df["timestamp"].dt.hour
    df["is_night"] = df["hour"].between(0, 5).astype(int)   # 00:00-05:59

    # Olay tipi özellikleri
    df["is_failed"] = (df["event"] == "Failed Login").astype(int)

    # IP bazlı: aynı IP'den kaç log gelmiş?
    df["ip_log_count"] = df.groupby("ip")["ip"].transform("count")

    # Kullanıcı bazlı: aynı kullanıcıdan kaç başarısız giriş?
    df["user_failed_count"] = df.groupby(["user", "is_failed"]).transform("size") if False else \
                              df.groupby("user")["is_failed"].transform("sum")

    return df

df = parse_logs(f"{BASE}/data/raw/synthetic_logs.csv")

print(" Parse edildi. Yeni özellikler:")
print(df[["timestamp","ip","event","attempts","hour","is_night","is_failed","ip_log_count","user_failed_count"]].head())

print("\n Saldırı satırlarından örnek:")
print(df[df["true_label"]==1].head(3))

# İşlenmiş veriyi kaydet (parquet daha hızlı ve küçük)
df.to_parquet(f"{BASE}/data/processed/logs.parquet", index=False)
print(f"\n Kaydedildi: {BASE}/data/processed/logs.parquet")

 Parse edildi. Yeni özellikler:
            timestamp            ip          event  attempts  hour  is_night  \
0 2026-11-17 00:01:00  192.168.1.31  Login Success         1     0         1   
1 2026-11-17 00:01:06  192.168.1.25  Login Success         2     0         1   
2 2026-11-17 00:02:16  192.168.1.15    File Access         3     0         1   
3 2026-11-17 00:02:28  192.168.1.36  Login Success         1     0         1   
4 2026-11-17 00:03:27  192.168.1.19  Login Success         3     0         1   

   is_failed  ip_log_count  user_failed_count  
0          0            82                  0  
1          0            76                100  
2          0            67                  0  
3          0            69                  0  
4          0            73                100  

 Saldırı satırlarından örnek:
              timestamp            ip         event   user  attempts  \
182 2026-11-17 02:00:06  192.168.1.45  Failed Login  admin        28   
185 2026-11-17 02:00:37 

In [5]:
# Hücre 4: Isolation Forest ile anomali tespiti
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

FEATURES = ["attempts", "is_failed", "is_night", "ip_log_count", "user_failed_count"]
X = df[FEATURES].values

# Ölçekleme: farklı birimleri aynı aralığa getir (attempts 1-40, is_night 0-1 vs.)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Model: contamination = verinin %5'inin anomali olduğunu varsay
model = IsolationForest(
    contamination=0.05,
    random_state=42,
    n_estimators=100
)
model.fit(X_scaled)

# Tahmin: -1 = anomali, +1 = normal
df["anomaly"] = model.predict(X_scaled)
df["anomaly_score"] = -model.score_samples(X_scaled)   # yüksek = daha anormal

print(" Model eğitildi.")
print(f" Tespit edilen anomali sayısı: {(df['anomaly']==-1).sum()} / {len(df)}")

# Drive'a kaydet
joblib.dump({"model": model, "scaler": scaler, "features": FEATURES},
            f"{BASE}/models/isoforest.pkl")
print(f" Model kaydedildi: {BASE}/models/isoforest.pkl")

# En anormal 5 kayıt
print("\n En anormal 5 kayıt:")
print(df.nlargest(5, "anomaly_score")[["timestamp","ip","event","attempts","is_night","anomaly_score","true_label"]])

 Model eğitildi.
 Tespit edilen anomali sayısı: 104 / 2100
 Model kaydedildi: /content/drive/MyDrive/SentinelAI/models/isoforest.pkl

 En anormal 5 kayıt:
              timestamp            ip         event  attempts  is_night  \
193 2026-11-17 02:01:51  192.168.1.45  Failed Login        39         1   
203 2026-11-17 02:04:06  192.168.1.45  Failed Login        39         1   
297 2026-11-17 02:24:27  192.168.1.45  Failed Login        39         1   
321 2026-11-17 02:28:13  192.168.1.45  Failed Login        39         1   
329 2026-11-17 02:29:55  192.168.1.45  Failed Login        39         1   

     anomaly_score  true_label  
193       0.723069           1  
203       0.723069           1  
297       0.723069           1  
321       0.723069           1  
329       0.723069           1  


In [6]:
# Hücre 5: Değerlendirme (gerçek etiketlerle karşılaştır)
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, roc_auc_score
)

# Model tahmini: anomaly=-1 → saldırı(1), anomaly=+1 → normal(0)
df["pred_label"] = (df["anomaly"] == -1).astype(int)

y_true = df["true_label"]
y_pred = df["pred_label"]
y_score = df["anomaly_score"]   # sürekli skor → ROC-AUC için

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print(" KARMAŞIKLIK MATRİSİ")
print(f"   TN={tn}  FP={fp}")
print(f"   FN={fn}  TP={tp}")
print()
print(" METRİKLER")
print(f"   Precision : {precision_score(y_true, y_pred):.3f}   (yakaladıklarımızın kaçı gerçekten saldırı?)")
print(f"   Recall    : {recall_score(y_true, y_pred):.3f}   (gerçek saldırıların kaçını yakaladık?)")
print(f"   F1        : {f1_score(y_true, y_pred):.3f}")
print(f"   ROC-AUC   : {roc_auc_score(y_true, y_score):.3f}")
print()
print(f" Yanlış Pozitif (FP) oranı : {fp / (fp+tn):.3f}   (normal logu yanlışlıkla saldırı dedik)")
print(f" Yanlış Negatif (FN) oranı : {fn / (fn+tp):.3f}   (saldırıyı kaçırdık)")

# Raporu Drive'a kaydet (portföyde önemli!)
report_path = f"{BASE}/reports/evaluation_v1.md"
with open(report_path, "w") as f:
    f.write(f"# SentinelAI – Anomaly Detection Değerlendirme Raporu (v1)\n\n")
    f.write(f"**Veri seti:** 2100 sentetik log (2000 normal, 100 saldırı)\n\n")
    f.write(f"**Model:** Isolation Forest (contamination=0.05)\n\n")
    f.write(f"## Karmaşıklık Matrisi\n\n")
    f.write(f"| | Tahmin: Normal | Tahmin: Saldırı |\n|---|---|---|\n")
    f.write(f"| **Gerçek: Normal** | {tn} | {fp} |\n")
    f.write(f"| **Gerçek: Saldırı** | {fn} | {tp} |\n\n")
    f.write(f"## Metrikler\n\n")
    f.write(f"- Precision: **{precision_score(y_true, y_pred):.3f}**\n")
    f.write(f"- Recall: **{recall_score(y_true, y_pred):.3f}**\n")
    f.write(f"- F1: **{f1_score(y_true, y_pred):.3f}**\n")
    f.write(f"- ROC-AUC: **{roc_auc_score(y_true, y_score):.3f}**\n")
    f.write(f"- False Positive Rate: **{fp/(fp+tn):.3f}**\n")

print(f"\n Rapor kaydedildi: {report_path}")

 KARMAŞIKLIK MATRİSİ
   TN=1988  FP=12
   FN=8  TP=92

 METRİKLER
   Precision : 0.885   (yakaladıklarımızın kaçı gerçekten saldırı?)
   Recall    : 0.920   (gerçek saldırıların kaçını yakaladık?)
   F1        : 0.902
   ROC-AUC   : 0.999

 Yanlış Pozitif (FP) oranı : 0.006   (normal logu yanlışlıkla saldırı dedik)
 Yanlış Negatif (FN) oranı : 0.080   (saldırıyı kaçırdık)

 Rapor kaydedildi: /content/drive/MyDrive/SentinelAI/reports/evaluation_v1.md
